In [17]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [18]:
# np.random.seed(42)

In [19]:
colunas=25
linhas=100

In [20]:
ativos = [a for a in range(colunas)]
retornos = np.random.normal(0, 0.1, size=(linhas,len(ativos)))

In [21]:
retornos.shape

(100, 25)

In [22]:
df = pd.DataFrame(retornos, index=range(linhas), columns=ativos)
df = df.pct_change().dropna().reset_index(drop=True)
df

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,-0.369007,5.560514,-1.495509,-0.094036,-0.252674,-0.241260,3.136028,3.334390,-0.788298,-1.749348,...,-3.025344,-2.347781,-2.037901,-1.224665,0.583192,-2.799045,0.592110,0.450843,-2.962138,-15.885621
1,-2.313072,-0.702778,-2.233421,9.199421,-0.510780,-0.857943,-0.338658,2.908984,2.645163,-3.242714,...,0.207439,-1.229377,1.586984,0.315134,-0.729449,-1.583935,1.144454,-1.433390,-1.650828,-1.015260
2,-0.438106,-0.429807,0.241286,-0.658752,0.964532,9.141226,-1.042493,-0.012792,-0.327115,-1.080745,...,-0.683372,-20.643918,-0.937458,0.243024,8.031919,-1.639935,-1.483823,0.290291,-2.951151,-137.953459
3,-1.225922,-6.272688,-1.032793,-3.456927,-2.782196,0.652174,-101.659433,-0.862720,-1.194587,-8.412907,...,-0.943124,-0.837928,-16.412411,1.637294,-0.569455,0.470302,2.349492,-1.319013,-0.696055,0.362938
4,-2.659506,-0.050210,3.681832,-1.034657,-0.993093,-0.996684,-0.610625,0.453292,-2.743633,-2.261183,...,44.510936,0.351068,0.395179,-0.458069,0.221129,2.059185,-1.046958,-5.120280,1.252775,-1.715007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,-2.587016,-0.973553,-0.997801,0.833378,-4.102657,-4.906223,-4.881539,-1.550262,-0.140295,3.672527,...,-0.160800,-0.859903,93.177805,-0.381582,-2.879025,-0.598414,-1.711200,-0.720915,-1.700592,-5.824045
95,0.117783,8.245794,-68.007208,-1.130479,-0.268154,-0.286811,-1.284706,-1.398130,0.496612,-3.536098,...,-3.060646,6.105062,-2.607169,-1.849483,0.587748,-5.810967,-2.876048,5.318040,-0.648281,1.775256
96,-2.145490,-1.454173,-0.873884,-3.148063,0.312889,-1.730788,-0.773158,3.055663,-2.242451,-1.570904,...,-0.878913,-0.137924,-0.521376,-1.412222,-0.398588,-0.769275,0.112441,-0.954380,-2.840594,-1.224649
97,0.265788,-5.173370,-7.352022,-4.336495,-0.742539,-2.165523,-4.180877,-0.638645,-2.349670,-0.207774,...,-6.895554,-0.254570,-1.794072,-13.180355,-2.707783,-4.313332,-1.624007,-19.647949,-3.263213,7.287156


In [23]:
df.cov()

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
0,288.227647,-3.148090,201.546850,-9.651752,4.691457,30.897915,-38.799718,0.337972,0.239747,0.708963,...,-22.502684,-3.269458,0.075337,-4.075243,2.346430,-3.417027,7.734443,-1.008560,-23.980632,-4.469381
1,-3.148090,607.639723,35.852477,4.082029,-3.672334,3.001528,-15.105844,3.861293,-9.078484,-1.888312,...,1.401644,37.385985,0.088347,-1.563312,-8.509368,11.072381,0.746494,-1.669877,19.231980,19.936888
2,201.546850,35.852477,13316.078491,15.037865,4.186424,-281.780260,46.708549,-35.094538,29.849806,1.475437,...,-80.393599,-249.801769,-0.870333,-3.693244,31.960012,-10.691908,13.196247,-28.345576,-34.016374,67.788699
3,-9.651752,4.082029,15.037865,64.335505,20.137722,-47.541971,-26.201690,-1.613578,2.779872,-4.843446,...,85.652708,-18.541824,2.727791,3.531441,3.701027,0.407337,0.102530,1.038119,-2.626833,-14.538727
4,4.691457,-3.672334,4.186424,20.137722,173.797749,-336.519186,-12.017035,-6.152494,7.583991,0.185676,...,-15.815633,35.293692,-3.046407,0.036310,-7.740103,-7.813713,-7.334037,-4.890035,-3.367805,0.628476
5,30.897915,3.001528,-281.780260,-47.541971,-336.519186,836.996883,5.433358,16.769089,-15.529368,0.292285,...,-19.460643,-37.001248,-4.578127,-5.516491,14.503894,-8.772318,25.566352,-1.532297,-8.870008,-25.398875
6,-38.799718,-15.105844,46.708549,-26.201690,-12.017035,5.433358,3505.977998,14.695960,-5.819400,381.750307,...,27.789892,43.333884,18.289309,7.154032,-26.937453,10.752442,-6.461978,3.872572,41.256373,49.608639
7,0.337972,3.861293,-35.094538,-1.613578,-6.152494,16.769089,14.695960,108.239649,2.568616,-4.956550,...,-5.014589,-6.741710,3.584286,-0.083400,9.484701,-1.378806,0.809190,-2.589219,-2.301528,-8.521241
8,0.239747,-9.078484,29.849806,2.779872,7.583991,-15.529368,-5.819400,2.568616,343.597360,6.504376,...,37.930596,88.731802,11.247768,-9.481978,48.220440,-25.247901,-1.857566,-3.336572,13.725645,-115.238158
9,0.708963,-1.888312,1.475437,-4.843446,0.185676,0.292285,381.750307,-4.956550,6.504376,151.721377,...,39.787442,15.696202,0.481574,-1.865925,-2.025145,4.954920,-1.119144,-2.307280,9.725953,5.571444


In [24]:
df.iloc[0,0]

np.float64(-0.3690073533499675)

In [25]:
model = pyo.ConcreteModel()

model.ativos = pyo.Set(initialize=ativos)
model.range_ativos = pyo.RangeSet(0, len(ativos)-1)
model.periodos = pyo.Set(initialize=range(linhas-1))
model.retornos = pyo.Param(model.range_ativos, initialize=lambda model,a: df.mean().iloc[a])
model.sigma = pyo.Param(model.range_ativos, model.range_ativos, initialize=lambda model,a,b: df.cov().iloc[a,b])
# model.x = pyo.Var(model.range_ativos, domain=pyo.NonNegativeReals)
model.y=pyo.Var(model.range_ativos, domain=pyo.Binary)
#linear sharpe
model.z = pyo.Var(model.range_ativos,domain=pyo.NonNegativeReals)
model.k = pyo.Var(domain=pyo.NonNegativeReals)

#Restrições do sharpe
def res_z(model):
    return sum(model.z[a] for a in model.range_ativos) == model.k
model.res_res_z = pyo.Constraint(rule=res_z)

def res_k(model):
    return sum(model.retornos[a] * model.z[a] for a in model.range_ativos) == 1
model.res_res_k = pyo.Constraint(rule=res_k)

def obj_rule(model):
    var = sum(model.sigma[a,b] * model.z[a] * model.z[b] for a in model.range_ativos for b in model.range_ativos)
    return var
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# restrição de quantidade minima de ativos selecionados
# def res_min_ativos(model,a):
#     return model.z[a] >= model.y[a]*0.01
# model.res_min_ativos = pyo.Constraint(model.range_ativos, rule=res_min_ativos)

# def res_max_ativos(model):
#     return sum(model.y[a] for a in model.range_ativos) >= 2
# model.res_max_ativoss = pyo.Constraint( rule=res_max_ativos)
# def res_max_ativos(model):
#     return sum(model.y[a] for a in model.range_ativos) <= 2
# model.res_max_ativoss = pyo.Constraint( rule=res_max_ativos)


In [26]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmpqc1tt3na.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmpawyhxukc.pyomo.lp' read.
Read time = 0.00 sec. (0.03 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmpawyhxukc.pyomo.lp
Objective sense      : Minimize
Variables            :      26  [Nneg: 26,  Qobj: 25]
Objective nonzeros   :       0
Objective Q nonzeros :     625
Linear constraints   :       2  [Equal: 2]
  Nonzeros           :      51
  RHS nonzeros       :       1

Variables            : Min LB: 0.000000         Max UB: all infi

In [29]:
print(pyo.value(model.k))
for a in model.ativos:
    # print(f'Ativo {a}: Z:{model.z[a].value:.4f};K:{model.k.value}; X=Z/K: {(model.z[a].value/model.k.value):.4f}; y={model.y[a].value:.4f}')
    print(f'Ativo {a}: Z:{model.z[a].value:.4f};K:{model.k.value}; X=Z/K: {(model.z[a].value/model.k.value):.4f}')

0.6739334364922813
Ativo 0: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 1: Z:0.0967;K:0.6739334364922813; X=Z/K: 0.1435
Ativo 2: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 3: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 4: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 5: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 6: Z:0.0305;K:0.6739334364922813; X=Z/K: 0.0453
Ativo 7: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 8: Z:0.0438;K:0.6739334364922813; X=Z/K: 0.0651
Ativo 9: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 10: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 11: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 12: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 13: Z:0.0554;K:0.6739334364922813; X=Z/K: 0.0821
Ativo 14: Z:0.2458;K:0.6739334364922813; X=Z/K: 0.3647
Ativo 15: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 16: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
Ativo 17: Z:0.0000;K:0.6739334364922813; X=Z/K: 0.0000
A

In [40]:
y_selec = {a: np.round((model.z[a].value)/model.k.value, 4) for a in model.range_ativos if (model.z[a].value)/model.k.value > 0}

In [53]:
final  = {z:x for z,x in y_selec.items() if x > 0.0}
final

{1: np.float64(0.1435),
 6: np.float64(0.0453),
 8: np.float64(0.0651),
 13: np.float64(0.0821),
 14: np.float64(0.3647),
 19: np.float64(0.2993)}

In [56]:
final.values()

dict_values([np.float64(0.1435), np.float64(0.0453), np.float64(0.0651), np.float64(0.0821), np.float64(0.3647), np.float64(0.2993)])

In [59]:
df[final.keys()]

,1,6,8,13,14,19
0,5.560514,3.136028,-0.788298,0.493991,-1.192692,0.583192
1,-0.702778,-0.338658,2.645163,0.571786,-5.037144,-0.729449
2,-0.429807,-1.042493,-0.327115,-1.246231,-3.282074,8.031919
3,-6.272688,-101.659433,-1.194587,7.007150,-0.827262,-0.569455
4,-0.050210,-0.610625,-2.743633,0.093046,-3.835263,0.221129
...,...,...,...,...,...,...
94,-0.973553,-4.881539,-0.140295,-0.551229,-0.228779,-2.879025
95,8.245794,-1.284706,0.496612,-2.817204,-0.078539,0.587748
96,-1.454173,-0.773158,-2.242451,-0.864229,-1.386081,-0.398588
97,-5.173370,-4.180877,-2.349670,-3.407349,-0.949612,-2.707783


In [57]:
df[final.keys()]*final.values()

,1,6,8,13,14,19
0,0.797934,0.142062,-0.051318,0.040557,-0.434975,0.174549
1,-0.100849,-0.015341,0.172200,0.046944,-1.837046,-0.218324
2,-0.061677,-0.047225,-0.021295,-0.102316,-1.196972,2.403953
3,-0.900131,-4.605172,-0.077768,0.575287,-0.301702,-0.170438
4,-0.007205,-0.027661,-0.178611,0.007639,-1.398720,0.066184
...,...,...,...,...,...,...
94,-0.139705,-0.221134,-0.009133,-0.045256,-0.083436,-0.861692
95,1.183271,-0.058197,0.032329,-0.231292,-0.028643,0.175913
96,-0.208674,-0.035024,-0.145984,-0.070953,-0.505504,-0.119297
97,-0.742379,-0.189394,-0.152964,-0.279743,-0.346323,-0.810439


In [51]:
pyo.value(model.obj)

32.25710643462363

### Adicionar cardinalidade e peso min

In [79]:
model = pyo.ConcreteModel()

model.ativos = pyo.Set(initialize=ativos)
model.range_ativos = pyo.RangeSet(0, len(ativos)-1)
model.periodos = pyo.Set(initialize=range(linhas-1))
model.retornos = pyo.Param(model.range_ativos, initialize=lambda model,a: df.mean().iloc[a])
model.sigma = pyo.Param(model.range_ativos, model.range_ativos, initialize=lambda model,a,b: df.cov().iloc[a,b])
# model.x = pyo.Var(model.range_ativos, domain=pyo.NonNegativeReals)
model.y=pyo.Var(model.range_ativos, domain=pyo.Binary)
#linear sharpe
model.z = pyo.Var(model.range_ativos,domain=pyo.NonNegativeReals)
model.k = pyo.Var(domain=pyo.NonNegativeReals)
model.w = pyo.Var(model.range_ativos, domain=pyo.NonNegativeReals)
model.K_MAX = pyo.Param(initialize=100)
model.VALOR_PM = pyo.Param(initialize=0.10)
#Restrições do sharpe
def res_z(model):
    return sum(model.z[a] for a in model.range_ativos) == model.k
model.res_res_z = pyo.Constraint(rule=res_z)

def res_k(model):
    return sum(model.retornos[a] * model.z[a] for a in model.range_ativos) == 1
model.res_res_k = pyo.Constraint(rule=res_k)

def obj_rule(model):
    var = sum(model.sigma[a,b] * model.z[a] * model.z[b] for a in model.range_ativos for b in model.range_ativos)
    return var
model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

# Restricao peso min
def r_peso_min(model,a):
    return model.w[a] <= model.K_MAX * model.y[a]
model.peso_min = pyo.Constraint(model.range_ativos,rule=r_peso_min)

def r_peso_min_2(model,a):
    return model.w[a] <= model.k
model.peso_min_2 = pyo.Constraint(model.range_ativos, rule=r_peso_min_2)

def r_peso_min_final(model,a):
    return model.w[a] >= model.k-model.K_MAX*(1-model.y[a])
model.peso_min_final = pyo.Constraint(model.range_ativos, rule=r_peso_min_final)

def pm(model,a):
    return model.z[a] >= model.VALOR_PM*model.w[a]
model.r_pm = pyo.Constraint(model.range_ativos, rule=pm)

def teto_sem_selecao(model, a):
    return model.z[a] <= model.w[a]
model.teto_sem_selecao = pyo.Constraint(model.range_ativos, rule=teto_sem_selecao)

# restrição cardinalidade
def res_min_ativos(model):
    return sum(model.y[a] for a in model.range_ativos) >= 8
model.res_min_ativoss = pyo.Constraint( rule=res_min_ativos)
def res_max_ativos(model):
    return sum(model.y[a] for a in model.range_ativos) <= 15
model.res_max_ativoss = pyo.Constraint( rule=res_max_ativos)




In [80]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmpkmlts_ay.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmps6pbek7t.pyomo.lp' read.
Read time = 0.02 sec. (0.04 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmps6pbek7t.pyomo.lp
Objective sense      : Minimize
Variables            :      76  [Nneg: 51,  Binary: 25,  Qobj: 25]
Objective nonzeros   :       0
Objective Q nonzeros :     625
Linear constraints   :     129  [Less: 126,  Greater: 1,  Equal: 2]
  Nonzeros           :     376
  RHS nonzeros       :      28

Variables            : Min

In [82]:
print(pyo.value(model.k))

for a in model.range_ativos:
    print(f'Ativo {a}:W: {model.w[a].value} Z:{model.z[a].value:.4f};K:{model.k.value}; X=Z/K: {(model.z[a].value/model.k.value):.4f}; y={model.y[a].value:.4f}')
    # print(f'Ativo {a}: Z:{model.z[a].value:.4f};K:{model.k.value}; X=Z/K: {(model.z[a].value/model.k.value):.4f}')

0.6722741169629103
Ativo 0:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=0.0000
Ativo 1:W: 0.6722741169629103 Z:0.0856;K:0.6722741169629103; X=Z/K: 0.1273; y=1.0000
Ativo 2:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 3:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 4:W: 0.6722741169629103 Z:0.0672;K:0.6722741169629103; X=Z/K: 0.1000; y=1.0000
Ativo 5:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 6:W: 0.6722741169629113 Z:0.0672;K:0.6722741169629103; X=Z/K: 0.1000; y=1.0000
Ativo 7:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 8:W: 0.6722741169629104 Z:0.0672;K:0.6722741169629103; X=Z/K: 0.1000; y=1.0000
Ativo 9:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 10:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.0000; y=-0.0000
Ativo 11:W: 0.6722741169629104 Z:0.0672;K:0.6722741169629103; X=Z/K: 0.1000; y=1.0000
Ativo 12:W: 0.0 Z:0.0000;K:0.6722741169629103; X=Z/K: 0.00